# Significant traces - all neurons

A standalone version of cell 13 in `General_analysis_single_fish_reusable.ipynb`.
Run top to bottom after selecting the fish and imaging rate. Review the plots before running the final save cell. The detector receives **every column** of the processed ΔF/F file: no reliability, activity, or raster-variance filtering. Any ROI exclusion during preprocessing is already part of that file.

Compare original ΔF/F, noise-centered ΔF/F, and binary detected-event frames with identical neuron/time order. Centering subtracts the fitted noise mean; it does not produce raw ΔF or change ΔF/F units. Event frames are transient detections, not inferred spikes.

All exports contain `all` in the filename and use `significant_traces_all`, separate from the existing filtered cache. Existing loaders do not automatically load these exports.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.significant_trace_detection import compute_noise_model_romano_fast_modular
from src.plotting_single_fish import raster_with_stimuli

## Settings

For Clara's data, change only `fish_id` below (for example, `LC001_f01` or `LC002_f03`) and run all cells. Each fish folder sits directly inside `main_path`; no experiment suffix is added. Input and output paths are built automatically.

`FPS_2P = 2.0` matches the preprocessing metadata checked for `LC001_f01`. Confirm this setting if another acquisition used a different rate. Detector settings match cell 13, including its 2200-bin override. `OUTPUT_DIR = None` saves under the selected fish's `03_analysis/functional/plots/significant_traces_all`.

In [ ]:
# Change only this to select another fish in Clara's folder.
fish_id = "LC001_f01"

# Common parent folder for all Clara fish.
main_path = Path("D:/Alejandro/Data/OneDrive - Universit\u00e9 de Lausanne/Lab/Data/others/Clara")

FPS_2P = 2.0
OUTPUT_DIR = None  # Automatically use the selected fish's plots folder.
SIG_SETTINGS = dict(
    n_bins=2200, k_neighbors=100, confCutOff=90,
    tauDecay=6.0, random_state=0, plot_odds=False, mode="current",
)
SAVE_OUTPUTS = True

## Load every input neuron

No cached raster or saved neuron-index selection is loaded. Column IDs are zero-based indices in this input file, retained in the saved arrays.

In [ ]:
# Build the standard merged input path from the fish ID.
analysis_dir = main_path / fish_id / "03_analysis" / "functional"
prefix = "_".join(fish_id.split("_")[:2])
DFOF_FILE = analysis_dir / "suite2P" / "merged_dFoF" / f"{prefix}_dFoF_merged.npy"
output_dir = analysis_dir / "plots" / "significant_traces_all" if OUTPUT_DIR is None else Path(OUTPUT_DIR)

if not DFOF_FILE.is_file():
    raise FileNotFoundError(f"Check fish_id and main_path; input file not found: {DFOF_FILE}")
if not np.isfinite(FPS_2P) or FPS_2P <= 0:
    raise ValueError("FPS_2P must be finite and positive.")
dfof_all = np.load(DFOF_FILE, allow_pickle=False)
if dfof_all.ndim != 2 or dfof_all.shape[0] < 3 or dfof_all.shape[1] == 0:
    raise ValueError(f"Expected (time >= 3, neurons >= 1); got {dfof_all.shape}.")
if not np.isfinite(dfof_all).all():
    raise ValueError("Input contains NaN/inf. Repair the upstream input; no columns are silently dropped.")
neuron_indices_all = np.arange(dfof_all.shape[1])
print(f"Loaded {dfof_all.shape[0]} frames - {dfof_all.shape[1]} neurons; all columns retained.")

## Detect significant events

Run the same canonical current-mode detector as cell 13, directly on `dfof_all`. Zero-event neurons remain in the result. Detection thresholds still distinguish events from noise; no neuron selection is applied.

In [ ]:
result_all = compute_noise_model_romano_fast_modular(
    dfof_all, fps=FPS_2P, **SIG_SETTINGS,
)
(map_of_odds_all, deltaF_center_all, density_data_all, density_noise_all,
 xev_all, yev_all, raster_all, joint_map_all) = result_all
assert raster_all.shape == deltaF_center_all.shape == dfof_all.shape
print(f"Retained {raster_all.shape[1]} neurons; {np.count_nonzero(raster_all)} detected event frames.")

## Compare ΔF/F and events for the whole population

The three heatmaps keep the original neuron order and the same time axis. Adjust the ΔF/F color limits here; values outside the display limits remain in the saved data. No neurons are removed for display.

In [ ]:
FIGSIZE_ALL = (12, 9)
VMIN_DFF = 0.0
VMAX_DFF = 0.05

fig_all, axes_all = plt.subplots(3, 1, figsize=FIGSIZE_ALL, sharex=True, constrained_layout=True)
for ax, values, label, binary in zip(
    axes_all,
    (dfof_all, deltaF_center_all, raster_all),
    ("Original ΔF/F - all neurons", "Centered ΔF/F - all neurons", "Detected event frames - all neurons"),
    (False, False, True),
):
    image = raster_with_stimuli(
        ax, values, fps=FPS_2P, fish_id=prefix,
        title_suffix=label, neuron_order=neuron_indices_all,
        is_binary=binary, vmin=0 if binary else VMIN_DFF,
        vmax=1 if binary else VMAX_DFF,
    )
    fig_all.colorbar(image, ax=ax, label="Event (0/1)" if binary else "ΔF/F")
plt.show()

## Extra comparison: sorted by peak original Delta F/F

Rank every neuron by its maximum **original** Delta F/F across the full recording, highest first. The same permutation is used for all three panels; no neurons are removed. Saved trace arrays retain their original order.

In [ ]:
FIGSIZE_SORTED = (12, 9)
VMIN_DFF_SORTED = 0.0
VMAX_DFF_SORTED = 0.05

peak_dfof_all = np.max(dfof_all, axis=0)
peak_order_all = np.argsort(-peak_dfof_all, kind="stable")
fig_sorted_all, axes_sorted_all = plt.subplots(
    3, 1, figsize=FIGSIZE_SORTED, sharex=True, constrained_layout=True,
)
for ax, values, label, binary in zip(
    axes_sorted_all,
    (dfof_all, deltaF_center_all, raster_all),
    ("Original Delta F/F", "Centered Delta F/F", "Detected event frames"),
    (False, False, True),
):
    image = raster_with_stimuli(
        ax, values, fps=FPS_2P, fish_id=prefix,
        title_suffix=f"{label} - all neurons, sorted by original peak",
        neuron_order=peak_order_all, is_binary=binary,
        vmin=0 if binary else VMIN_DFF_SORTED,
        vmax=1 if binary else VMAX_DFF_SORTED,
    )
    ax.set_ylabel("Neuron rank (highest peak first)")
    fig_sorted_all.colorbar(image, ax=ax, label="Event (0/1)" if binary else "Delta F/F")
plt.show()

## Inspect one neuron

Edit `NEURON_INDEX` (original zero-based column ID) and rerun this cell. Red dots mark every detected event frame on the original ΔF/F trace, including sustained runs. `TIME_WINDOW_S = None` shows the full recording; a pair such as `(100, 200)` zooms in without changing detection or exports.

In [ ]:
NEURON_INDEX = 0
TIME_WINDOW_S = None
FIGSIZE_NEURON = (12, 3.5)

if not 0 <= NEURON_INDEX < dfof_all.shape[1]:
    raise ValueError(f"NEURON_INDEX must be in 0..{dfof_all.shape[1] - 1}.")
time_s = np.arange(dfof_all.shape[0]) / FPS_2P
events = raster_all[:, NEURON_INDEX].astype(bool)
fig_neuron_all, ax = plt.subplots(figsize=FIGSIZE_NEURON, constrained_layout=True)
ax.plot(time_s, dfof_all[:, NEURON_INDEX], lw=0.8, label="Original ΔF/F")
ax.plot(time_s, deltaF_center_all[:, NEURON_INDEX], lw=0.8, alpha=0.75, label="Centered ΔF/F")
ax.scatter(time_s[events], dfof_all[events, NEURON_INDEX], s=12, c="tab:red", label="Detected event frames", zorder=3)
ax.set(xlabel="Time (s)", ylabel="ΔF/F", title=f"{prefix} - neuron {NEURON_INDEX} (all-neuron detector)")
if TIME_WINDOW_S is not None:
    ax.set_xlim(*TIME_WINDOW_S)
ax.legend(loc="upper right")
plt.show()

## Random examples from highly active neurons

Here "highly active" means the **top 10% by peak original Delta F/F**, restricted to positive peaks. This selects examples for display only; detection and saved population arrays still include all neurons. Draw up to 20 distinct examples without replacement. Change `HIGH_ACTIVITY_FRACTION`, `N_EXAMPLES`, or `EXAMPLE_SEED` below. Panel titles give original zero-based column IDs. Red dots mark detected event frames on the original trace.

In [ ]:
N_EXAMPLES = 20
HIGH_ACTIVITY_FRACTION = 0.10
EXAMPLE_SEED = 42
FIGSIZE_EXAMPLES = (16, 15)
TIME_WINDOW_EXAMPLES_S = None  # e.g. (100, 200); display only

if N_EXAMPLES < 1 or not 0 < HIGH_ACTIVITY_FRACTION <= 1:
    raise ValueError("Use N_EXAMPLES >= 1 and 0 < HIGH_ACTIVITY_FRACTION <= 1.")
n_high = max(1, int(np.ceil(HIGH_ACTIVITY_FRACTION * dfof_all.shape[1])))
high_activity_pool_all = peak_order_all[:n_high]
high_activity_pool_all = high_activity_pool_all[peak_dfof_all[high_activity_pool_all] > 0]
example_indices_all = np.random.default_rng(EXAMPLE_SEED).choice(
    high_activity_pool_all, size=min(N_EXAMPLES, len(high_activity_pool_all)), replace=False,
)
fig_examples_all, axes_examples_all = plt.subplots(
    max(1, int(np.ceil(len(example_indices_all) / 4))), 4,
    figsize=FIGSIZE_EXAMPLES, squeeze=False, constrained_layout=True,
)
for ax in axes_examples_all.flat:
    ax.set_visible(False)
for ax, neuron in zip(axes_examples_all.flat, example_indices_all):
    ax.set_visible(True)
    events = raster_all[:, neuron].astype(bool)
    ax.plot(time_s, dfof_all[:, neuron], lw=0.7, label="Original Delta F/F")
    ax.plot(time_s, deltaF_center_all[:, neuron], lw=0.7, alpha=0.7, label="Centered Delta F/F")
    ax.scatter(time_s[events], dfof_all[events, neuron], s=6, c="tab:red", label="Event frames", zorder=3)
    ax.set(title=f"Neuron {neuron} | peak {peak_dfof_all[neuron]:.3g}", xlabel="Time (s)", ylabel="Delta F/F")
    if TIME_WINDOW_EXAMPLES_S is not None:
        ax.set_xlim(*TIME_WINDOW_EXAMPLES_S)
if len(example_indices_all):
    axes_examples_all.flat[0].legend(fontsize=7, loc="best")
else:
    fig_examples_all.suptitle("No neurons with positive original Delta F/F peaks")
print(f"Showing {len(example_indices_all)} random examples from {len(high_activity_pool_all)} high-activity neurons.")
plt.show()

## Save outputs - run this last, after reviewing plots

This is the only cell that writes files. It creates the output directory before saving, including when you rerun it separately. Set `SAVE_OUTPUTS = False` to skip all exports.

Trace arrays remain in original column order. The NPZ includes the display sort order, example IDs, and example-selection settings so the figures can be traced back to original neuron IDs. All filenames contain `all`.

In [ ]:
out_all = output_dir / f"{prefix}_significant_traces_all.npz"
if SAVE_OUTPUTS:
    output_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_all, dfof=dfof_all, deltaF_center=deltaF_center_all,
        raster=raster_all, neuron_indices=neuron_indices_all, fps=FPS_2P,
        source_file=str(DFOF_FILE.resolve()), settings_json=json.dumps(SIG_SETTINGS),
        peak_order=peak_order_all, example_indices=example_indices_all,
        example_settings_json=json.dumps(dict(
            n_examples=N_EXAMPLES, high_activity_fraction=HIGH_ACTIVITY_FRACTION,
            seed=EXAMPLE_SEED, time_window_s=TIME_WINDOW_EXAMPLES_S,
        )),
    )
    for figure, filename in (
        (fig_all, f"{prefix}_dfof_raster_comparison_all.png"),
        (fig_sorted_all, f"{prefix}_dfof_raster_peak_sorted_all.png"),
        (fig_neuron_all, f"{prefix}_neuron_{NEURON_INDEX}_comparison_all.png"),
        (fig_examples_all, f"{prefix}_high_activity_examples_all.png"),
    ):
        figure.savefig(output_dir / filename, dpi=180)
    print(f"Saved arrays and 4 comparison figures to {output_dir}")
else:
    print("SAVE_OUTPUTS is False; no files were saved.")